In [2]:
import chess
import random
import json
from tqdm import tqdm

OUTPUT_FILE = "synthetic_chess_dataset.jsonl"

In [3]:
def generate_templates(board, move, san) -> list[str]:
    piece = board.piece_at(move.from_square)
    target = chess.square_name(move.to_square)

    is_capture = board.is_capture(move)
    is_castle = board.is_castling(move)
    is_promotion = move.promotion is not None

    templates = []

    file_name = chess.FILE_NAMES[chess.square_file(move.from_square)]
    
    if is_castle:    # todo: check if this also works for black
        if chess.square_file(move.to_square) == 6:
            templates += [
                "Castle kingside",
                "Kingside castle",
                "Perform kingside castling",
                "King castles short",
                "Castle short",
                "Short castle", 
            ]
        else:
            templates += [
                "Castle queenside",
                "Queenside castle",
                "Perform queenside castling",
                "King castles long",
                "Castle long",
                "Long castle",
            ]
        return templates

    if is_promotion:
        promo_piece = chess.piece_name(move.promotion)
        templates += [
            f"Promote pawn to {promo_piece} on {target}",
            f"Advance pawn to {target} and promote to {promo_piece}",
            f"Move pawn to {target} and make it a {promo_piece}",
        ]
        return templates

    p_name = chess.piece_name(piece.piece_type)

    if is_capture:
        templates += [
            f"{p_name.capitalize()} captures on {target}",
            f"Capture the piece on {target} with the {p_name}",
            f"Take on {target} using the {p_name}",
            f"Use the {p_name} to capture on {target}",
        ]
    elif piece.piece_type == chess.PAWN:
        templates += [
            f"Advance pawn to {target}",
            f"Push the pawn to {target}",
            f"Move pawn to {target}",
            f"Advance {file_name} pawn",
        ]
    else:
        templates += [
            f"Move the {p_name} to {target}",
            f"Play {p_name} to {target}",
            f"Develop the {p_name} to {target}",
            f"{p_name.capitalize()} goes to {target}",
        ]

    templates.append(f"Play {san}")

    return templates

In [7]:
def generate_dataset(board):
    print(board)
    with open(OUTPUT_FILE, "w") as f:
        dataset = []
        legal_moves = list(board.legal_moves)
        for move in legal_moves:
            san = board.san(move)
            templates = generate_templates(board, move, san)
            for template in templates:
                dataset.append(
                    {
                        "san": san,
                        "desc": template
                    }
                )
            
            for data in dataset:
                f.write(json.dumps(data) + "\n")

In [8]:
if __name__ == "__main__":
    board = chess.Board("rnb1k1nr/p2p1ppp/5q2/1pbN1N1P/4PBP1/3P1Q2/PPP5/R4KR1 b kq - 4 17")
    generate_dataset(board)
    print("Dataset generation complete.")

r n b . k . n r
p . . p . p p p
. . . . . q . .
. p b N . N . P
. . . . P B P .
. . . P . Q . .
P P P . . . . .
R . . . . K R .
Dataset generation complete.


In [11]:
testing_data = []
testing_data.append(("Capture the pawn on the queenside with my knight, centralizing it while eliminating a defender.", "Nxc7+"))
testing_data.append(("Capture the bishop on the queenside with my knight, removing a key attacker near my position.", "Nxb5"))
testing_data.append(("Bring my knight back toward the center to reinforce key central squares and improve coordination.", "Ne3"))
testing_data.append(("Retreat my knight to a safer square closer to my king, consolidating the position.", "Ng3"))
testing_data.append(("Advance my kingside pawn to challenge the opposing queen and gain space with tempo.", "g5"))
testing_data.append(("Push my advanced rook pawn further, increasing pressure on the kingside structure.", "h6"))
testing_data.append(("Advance my a-pawn one square to begin expanding on the queenside.", "a3"))
testing_data.append(("Advance my b-pawn one square to support potential queenside expansion.", "b3"))
testing_data.append(("Push my c-pawn forward to open lines and contest central control.", "c3"))
testing_data.append(("Centralize my rook by sliding it to the open central file, increasing pressure and coordination.", "Re1"))

In [13]:
import chess
import chess.engine
from sentence_transformers import SentenceTransformer, util
import torch

class NLChessInterpreter:
    def __init__(self, model_name="all-MiniLM-L6-v2", top_k=3):
        self.model = SentenceTransformer(model_name)
        self.top_k = top_k

    def describe_move(self, board, move):
        """
        Convert a move into a semantic description.
        """
        piece = board.piece_at(move.from_square)
        piece_name = chess.piece_name(piece.piece_type)

        from_sq = chess.square_name(move.from_square)
        to_sq = chess.square_name(move.to_square)

        color = "White" if piece.color == chess.WHITE else "Black"

        description = f"{color} {piece_name} from {from_sq} to {to_sq}"

        if board.is_capture(move):
            description += " capturing a piece"

        # Positional heuristics
        file = chess.square_file(move.to_square)
        if file <= 2:
            description += " on the queenside"
        elif file >= 5:
            description += " on the kingside"
        else:
            description += " in the center"

        # Pawn-specific heuristic
        if piece.piece_type == chess.PAWN:
            description += " advancing a pawn"

        return description

    def interpret(self, instruction, fen):
        board = chess.Board(fen)

        legal_moves = list(board.legal_moves)
        if not legal_moves:
            raise ValueError("No legal moves in this position.")

        descriptions = [
            self.describe_move(board, move) for move in legal_moves
        ]

        instruction_embedding = self.model.encode(
            instruction, convert_to_tensor=True
        )
        move_embeddings = self.model.encode(
            descriptions, convert_to_tensor=True
        )

        scores = util.cos_sim(
            instruction_embedding, move_embeddings
        )[0]

        # Get top-k indices
        top_k = min(self.top_k, len(legal_moves))
        top_results = torch.topk(scores, k=top_k)

        results = []
        for score, idx in zip(top_results.values, top_results.indices):
            move = legal_moves[idx]
            san = board.san(move)
            results.append((san, float(score)))

        return results


if __name__ == "__main__":
    interpreter = NLChessInterpreter()

    for i in testing_data:
        instruction = i[0]
        fen = "rnb1k1nr/p2p1ppp/5q2/1pbN1N1P/4PBP1/3P1Q2/PPP5/R4KR1 w kq - 4 17"
    
        move_san = interpreter.interpret(instruction, fen)
        print("Predicted Move:", move_san)
        print("Actual Move:", i[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Predicted Move: [('b3', 0.6198492050170898), ('b4', 0.6090743541717529), ('c3', 0.6033549308776855)]
Actual Move: Nxc7+
Predicted Move: [('Bxb8', 0.642782986164093), ('Bc7', 0.5793914794921875), ('Bc1', 0.5717929005622864)]
Actual Move: Nxb5
Predicted Move: [('Nd4', 0.4928327798843384), ('Nfe3', 0.4923310875892639), ('Ng3', 0.48559069633483887)]
Actual Move: Ne3
Predicted Move: [('Nh4', 0.5295568108558655), ('Nh6', 0.51510089635849), ('Nxf6+', 0.4993971884250641)]
Actual Move: Ng3
Predicted Move: [('b3', 0.5501792430877686), ('b4', 0.5293356776237488), ('h6', 0.5288417339324951)]
Actual Move: g5
Predicted Move: [('h6', 0.5563836693763733), ('c3', 0.5323141813278198), ('Rh1', 0.5301353335380554)]
Actual Move: h6
Predicted Move: [('a4', 0.6515897512435913), ('a3', 0.633777916431427), ('b4', 0.6060014963150024)]
Actual Move: a3
Predicted Move: [('b4', 0.6880654096603394), ('b3', 0.6789790391921997), ('a4', 0.6162149906158447)]
Actual Move: b3
Predicted Move: [('c3', 0.539086103439331), ('